# Explore your result

This notebook takes the `.h5ad` from
[01-first-prediction.ipynb](01-first-prediction.ipynb) and explores it on your
tissue:

1. where a gene's signal appears across the section;
2. a UMAP of the predicted expression;
3. clusters, drawn back on the slide.

The result is an ordinary `AnnData`, so everything here is `scanpy` and
`anndata`.

In [ ]:
from pathlib import Path

import anndata
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PREDICTION = ROOT / "data" / "prediction.h5ad"

if not PREDICTION.exists():
    raise FileNotFoundError(f"data/{PREDICTION.name}: run 01-first-prediction.ipynb first, it writes this file")

sc.settings.set_figure_params(dpi=110, facecolor="white", frameon=False)

adata = anndata.read_h5ad(PREDICTION)
print(adata)
print("model:", adata.uns["model"]["id"])

## 1. Find a gene

The gene axis is indexed by Ensembl identifier, and the symbol sits beside it
in `var["feature_name"]`. Symbols are renamed, retired and occasionally
reassigned; an identifier is not. So turn the symbols you hold into
identifiers once, and use the symbol only as a label. A previous symbol or an
alias resolves through `client.resolve_genes([...])`.

In [ ]:
symbol_to_id = dict(zip(adata.var["feature_name"], adata.var_names))
id_to_symbol = dict(zip(adata.var_names, adata.var["feature_name"]))

print("MS4A1 ->", symbol_to_id.get("MS4A1"))

## 2. Where a gene's signal appears

This section is lung cancer with tertiary lymphoid structures: organised immune
aggregates, B-cell follicles with T cells beside them, visible on the H&E. These
six genes mark compartments you can check against the stained section yourself.

| Gene | Marks |
|---|---|
| `MS4A1` | B cells |
| `CD79A` | B cells |
| `CD3E` | T cells |
| `PTPRC` | Immune cells |
| `COL1A1` | Collagen in the stroma |
| `EPCAM` | Epithelium |

A gene the model never saw measured is blank, and its `measured_in_training` is
`False`.

In [ ]:
PANEL = ["MS4A1", "CD79A", "CD3E", "PTPRC", "COL1A1", "EPCAM"]

missing = [s for s in PANEL if s not in symbol_to_id]
if missing:
    print("not in this result:", missing)
PANEL = [s for s in PANEL if s in symbol_to_id]
panel_ids = [symbol_to_id[s] for s in PANEL]

print(adata.var.loc[panel_ids, ["feature_name", "measured_in_training"]])

In [ ]:
def draw(values, title, ax, cmap="magma", clip=(1, 99), size=6):
    """One gene, or any value per spot, over the tissue."""
    xy = adata.obsm["spatial"]
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    lo, hi = np.percentile(finite, clip) if finite.size else (0, 1)
    handle = ax.scatter(xy[:, 0], xy[:, 1], c=values, s=size, cmap=cmap,
                        vmin=lo, vmax=hi, linewidths=0)
    ax.set_title(title, fontsize=10)
    ax.set_aspect("equal")
    ax.invert_yaxis()  # image rows increase downwards
    ax.set_axis_off()
    return handle


fig, axes = plt.subplots(2, 3, figsize=(15, 9.5))
for ax in axes.ravel()[len(PANEL):]:
    ax.set_axis_off()
for ax, symbol, gene_id in zip(axes.ravel(), PANEL, panel_ids):
    handle = draw(np.asarray(adata[:, gene_id].X).ravel(), symbol, ax)
    fig.colorbar(handle, ax=ax, fraction=0.046, shrink=0.8)
fig.suptitle("Predicted spatial gene expression", y=0.98)
fig.tight_layout()
plt.show()

Read the maps against each other and against the slide: `MS4A1` and `CD79A`
both mark B cells, while `COL1A1` and `EPCAM` mark the stroma and the
epithelium.

## 3. A UMAP of the predicted expression

Each point is one spot, placed by how alike its predicted expression is to
the others'. Blank genes are left out first.

In [ ]:
measured = adata.var["measured_in_training"].to_numpy(dtype=bool)
finite = np.isfinite(np.asarray(adata.X)).all(axis=0)
work = adata[:, measured & finite].copy()
print("genes in the analysis:", f"{work.n_vars:,}")

sc.pp.highly_variable_genes(work, n_top_genes=2000)
sc.tl.pca(work, n_comps=50, mask_var="highly_variable")
sc.pp.neighbors(work, n_neighbors=15, n_pcs=30)
sc.tl.umap(work)

In [ ]:
shown = [g for g in panel_ids if g in work.var_names]
sc.pl.umap(
    work,
    color=shown,
    title=[id_to_symbol[g] for g in shown],
    cmap="magma",
    ncols=3,
    frameon=False,
    show=True,
)

## 4. Clusters, on the slide

A UMAP shows which spots are alike. Drawn back on the slide, the clusters show
where they are.

In [ ]:
sc.tl.leiden(work, resolution=0.6, flavor="igraph", n_iterations=2, directed=False)
print(work.obs["leiden"].value_counts().sort_index().to_string())

sc.pl.umap(work, color="leiden", legend_loc="on data", frameon=False,
           title="Leiden clusters of predicted expression", show=True)

In [ ]:
clusters = work.obs["leiden"]
palette = plt.get_cmap("tab20")
xy = work.obsm["spatial"]

fig, ax = plt.subplots(figsize=(7.5, 7.5))
for index, label in enumerate(clusters.cat.categories):
    at = (clusters == label).to_numpy()
    ax.scatter(xy[at, 0], xy[at, 1], s=6, color=palette(index % 20), label=label,
               linewidths=0)
ax.set_aspect("equal")
ax.invert_yaxis()
ax.set_axis_off()
ax.set_title("The same clusters, on the slide")
ax.legend(markerscale=3, fontsize=8, loc="center left", bbox_to_anchor=(1, 0.5),
          frameon=False, title="leiden")
fig.tight_layout()
plt.show()

To learn what sets a cluster apart, rank its genes against the rest, and read
the identifiers back as symbols.

In [ ]:
sc.tl.rank_genes_groups(work, "leiden", method="wilcoxon")

for label in work.obs["leiden"].cat.categories:
    top = [id_to_symbol.get(g, g) for g in work.uns["rank_genes_groups"]["names"][label][:8]]
    print(f"cluster {label}: {', '.join(top)}")

## Next

- **Your own slide.** In notebook 01, set `slide_path` to your file. The rest
  is the same.
- **A bulk RNA profile.** Add the sample's bulk RNA profile to the prediction
  for its slide, and keep the same genes and file format:
  [Add a bulk RNA profile](https://docs.auroraomics.org/guides/bulk-rna/).
- **Aurora Intelligence.** To explore a result as a map, submit with
  `land_in_workspace=True` and import it into a workspace at
  <https://app.auroraomics.org>.
- **Research applications.**
  [Explore research applications](https://auroraomics.org/use-cases) on the
  website.